# Model 3 - Food ID test

In [1]:
!pip install transformers accelerate pillow -q

# Imports and configuration

In [2]:
from pathlib import Path
from PIL import Image
import torch
from transformers import pipeline as hf_pipeline

## Settings

In [3]:
# Path where the model will be cached after first download.
# In the app this is models/nateraw_food_cache/
# For this test, use a local Colab path.

CACHE_DIR = Path("/content/model3_cache")
CACHE_DIR.mkdir(parents = True, exist_ok = True)

In [16]:
# HuggingFace model identifier
MODEL_NAME = "nateraw/food"

# How many top predictions to return
TOP_K = 5

# Minimum confidence to accept a prediction as valid
MIN_SCORE = 0.30

# Your container crop images — update these paths
# These should be the tight crops of container interiors,
# equivalent to what Stage III produces at inference time.
CROP_IMAGE_PATHS = [
    "/content/crop_images/20260316_175134.jpg",
    "/content/crop_images/20260316_175744.jpg",
    "/content/crop_images/20260316_180309.jpg",
    "/content/crop_images/20260316_212838.jpg",
    "/content/crop_images/20260316_213414.jpg",
    "/content/crop_images/20260316_213957.jpg"
]

# Food class mapping — mirrors config.py FOOD_CLASS_MAP
# Maps Food-101 class names → your canonical food names
FOOD_CLASS_MAP = {
    "rice":         "rice",
    "fried_rice":   "rice",
    "risotto":      "rice",
    "paella":       "rice",
    "chicken_curry":"rice",
    "lentil_soup":  "lentils",
    "lentils":      "lentils",
    "falafel":      "lentils",
    "hummus":       "lentils",
    "bibimbap":     "lentils",
    "edamame":      "lentils",
    "beet_salad":   "lentils"
}
SUPPORTED_FOODS = ["rice", "lentils"]

# Load model

In [17]:
print(f"Loading {MODEL_NAME}...")
print(f"Cache directory: {CACHE_DIR}")
print("This will download ~350 MB on first run, then load from cache.\n")

# Using model_kwargs to pass the cache_dir to the underlying model/tokenizer
classifier = hf_pipeline(
    "image-classification",
    model = MODEL_NAME,
    top_k = TOP_K,
    model_kwargs = {"cache_dir": str(CACHE_DIR)}
)

print(f"Model loaded. Device: {classifier.device}")
print(f"Model type: {type(classifier.model).__name__}\n")

Loading nateraw/food...
Cache directory: /content/model3_cache
This will download ~350 MB on first run, then load from cache.



Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model loaded. Device: cpu
Model type: ViTForImageClassification



# Inspect Food-101 class labels

In [18]:
# Print all 101 class labels so you can identify which ones are relevant
# to your foods and add them to FOOD_CLASS_MAP if needed.

all_labels = sorted(classifier.model.config.id2label.values())
print(f"Food-101 has {len(all_labels)} classes:\n")
print("\n".join(f"  {l}" for l in all_labels))

# Highlight classes that are relevant to rice and lentils
print("\n--- Classes potentially relevant to your foods ---")
keywords = ["rice", "lentil", "dal", "dhal", "porridge", "grain", "pilaf",
            "risotto", "biryani", "congee", "fried_rice", "paella", "bibimbap",
            "miso_soup", "falafel", "hummus"]

for label in all_labels:
    if any(kw in label.lower() for kw in keywords):
        mapped = FOOD_CLASS_MAP.get(label.lower().replace(" ", "_"), "NOT MAPPED")
        print(f"  {label:30s} → {mapped}")

Food-101 has 101 classes:

  apple_pie
  baby_back_ribs
  baklava
  beef_carpaccio
  beef_tartare
  beet_salad
  beignets
  bibimbap
  bread_pudding
  breakfast_burrito
  bruschetta
  caesar_salad
  cannoli
  caprese_salad
  carrot_cake
  ceviche
  cheese_plate
  cheesecake
  chicken_curry
  chicken_quesadilla
  chicken_wings
  chocolate_cake
  chocolate_mousse
  churros
  clam_chowder
  club_sandwich
  crab_cakes
  creme_brulee
  croque_madame
  cup_cakes
  deviled_eggs
  donuts
  dumplings
  edamame
  eggs_benedict
  escargots
  falafel
  filet_mignon
  fish_and_chips
  foie_gras
  french_fries
  french_onion_soup
  french_toast
  fried_calamari
  fried_rice
  frozen_yogurt
  garlic_bread
  gnocchi
  greek_salad
  grilled_cheese_sandwich
  grilled_salmon
  guacamole
  gyoza
  hamburger
  hot_and_sour_soup
  hot_dog
  huevos_rancheros
  hummus
  ice_cream
  lasagna
  lobster_bisque
  lobster_roll_sandwich
  macaroni_and_cheese
  macarons
  miso_soup
  mussels
  nachos
  omelette
  oni

# Run inference on your crop images

In [19]:
if not CROP_IMAGE_PATHS:
    print("No images specified in CROP_IMAGE_PATHS.")
    print("Add paths to your container crop images and re-run this cell.")
else:
    for img_path in CROP_IMAGE_PATHS:
        img_path = Path(img_path)
        if not img_path.exists():
            print(f"File not found: {img_path}")
            continue

        print(f"\n{'='*60}")
        print(f"Image: {img_path.name}")
        print(f"{'='*60}")

        img = Image.open(img_path).convert("RGB")
        print(f"Size: {img.size[0]}×{img.size[1]} px")

        # Run inference
        predictions = classifier(img)

        # Display top-k results
        print(f"\nTop-{TOP_K} predictions:")
        for i, pred in enumerate(predictions):
            raw_label = pred["label"]
            score     = pred["score"]
            norm_key  = raw_label.lower().replace(" ", "_")
            mapped    = FOOD_CLASS_MAP.get(norm_key, "NOT MAPPED")
            bar       = "█" * int(score * 30)
            print(f"  {i+1}. {raw_label:30s}  {score:.3f}  {bar}  → {mapped}")

        # Determine final food type
        best = next(
            (p for p in predictions
             if FOOD_CLASS_MAP.get(
                 p["label"].lower().replace(" ", "_")
             ) in SUPPORTED_FOODS
             and p["score"] >= MIN_SCORE),
            None,
        )

        print()
        if best is None:
            print(f"  RESULT: ambiguous — no supported food found above "
                  f"threshold {MIN_SCORE}")
            print(f"  ACTION: manual override UI will be shown in the app")
        else:
            mapped_food = FOOD_CLASS_MAP[
                best["label"].lower().replace(" ", "_")
            ]
            print(f"  RESULT: '{mapped_food}'  "
                  f"(from '{best['label']}', score {best['score']:.3f})")
            if best["score"] < 0.60:
                print(f"  NOTE: confidence is low — consider adding more "
                      f"Food-101 label mappings to FOOD_CLASS_MAP")


Image: 20260316_175134.jpg
Size: 2541×1508 px

Top-5 predictions:
  1. fried_rice                      0.778  ███████████████████████  → rice
  2. risotto                         0.021    → rice
  3. pho                             0.017    → NOT MAPPED
  4. ramen                           0.015    → NOT MAPPED
  5. paella                          0.009    → rice

  RESULT: 'rice'  (from 'fried_rice', score 0.778)

Image: 20260316_175744.jpg
Size: 2572×1542 px

Top-5 predictions:
  1. fried_rice                      0.381  ███████████  → rice
  2. risotto                         0.111  ███  → rice
  3. macaroni_and_cheese             0.068  ██  → NOT MAPPED
  4. paella                          0.029    → rice
  5. ramen                           0.025    → NOT MAPPED

  RESULT: 'rice'  (from 'fried_rice', score 0.381)
  NOTE: confidence is low — consider adding more Food-101 label mappings to FOOD_CLASS_MAP

Image: 20260316_180309.jpg
Size: 2548×1514 px

Top-5 predictions:
  1. fried_

> As the model is robust, it is necessary to add the "lentils" label. For that end, the model should be fine-tuned.

# Cache verification

In [15]:
import time

print("Verifying cache...")
cache_files = list(CACHE_DIR.rglob("*"))
total_mb    = sum(f.stat().st_size for f in cache_files if f.is_file()) / 1e6
print(f"  Cache directory : {CACHE_DIR}")
print(f"  Files cached    : {len(cache_files)}")
print(f"  Total size      : {total_mb:.1f} MB")

print("\nWarm load time (from cache):")
t0 = time.time()
_ = hf_pipeline(
    "image-classification",
    model = MODEL_NAME,
    top_k = TOP_K,
    model_kwargs = {"cache_dir": str(CACHE_DIR)}
)
print(f"  {time.time() - t0:.2f}s")

print("\nCache test complete.")
print("If load time is under 5s, the cache is working correctly.")
print("Copy CACHE_DIR contents to models/nateraw_food_cache/ in the app repo.")

Verifying cache...
  Cache directory : /content/model3_cache
  Files cached    : 26
  Total size      : 1374.2 MB

Warm load time (from cache):


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

  0.69s

Cache test complete.
If load time is under 5s, the cache is working correctly.
Copy CACHE_DIR contents to models/nateraw_food_cache/ in the app repo.
